# Task 3 — soft mixture of experts initialized from your Task 2 models
Select **Runtime → Change runtime type → GPU**, then **Run all**. Allow Drive access and upload **task3_bundle.zip** at the prompt. This bundle already includes your four trained Task 2 best checkpoints; the large Task 2 results ZIP is not needed.

The gate starts from your trained classifier, and all three experts start from your trained specialists. Reconstruction is a differentiable blend of the unchanged input, noise expert, blur expert and occlusion expert. The gate uses temperature-scaled softmax; outputs are four weights and one restored image.

Stage 1 freezes experts (including dropout) and trains only the gate. Stage 2 unfreezes all experts and jointly fine-tunes at a lower learning rate. Training uses L1, SSIM, corruption cross-entropy and a batch-average usage-balance regularizer. Exactly balanced batches keep the balance target meaningful; the regularizer does not demand equal weights for each image.

Starting budgets: **6 trials × (1 warm-up + 3 joint epochs)**, then **3 warm-up + 20 joint epochs** for the selected configuration. Every trial and the final run initialize from the same Task 2 checkpoints, not from a previous trial. Best final checkpoint selection uses only joint-stage epochs, with a separate warm-up best retained for comparison.

Backups: `MyDrive/GenAI_Ass1/task3_v1`. After reconnecting, rerun cells with the same bundle and budgets to restore studies and resume final training. Validation uses the original fixed cases and the identical Task 2 severity sweep. No test evaluation runs. Mild-blur preservation, clean-image changes, gate usage and per-condition hard-vs-soft scores must be inspected before claiming improvement.

The final cell downloads **task3_results.zip**.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, zipfile, shutil, subprocess, sys, json, sqlite3
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime, then reconnect before training.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PROJECT = Path('/content/ass1_task3')
BACKUP = Path('/content/drive/MyDrive/GenAI_Ass1/task3_v1')
PROJECT.mkdir(exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)


In [ ]:
uploaded = files.upload()  # Select colab/task3_bundle.zip
if len(uploaded) != 1:
    raise ValueError('Upload exactly the generated source ZIP.')
archive_name = next(iter(uploaded))
with zipfile.ZipFile(archive_name) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Invalid ZIP path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# Keep Colab's CUDA-enabled PyTorch installation; do not install the CPU wheel.
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

config = json.loads(Path('configs/task3.json').read_text())
print('Task 3 configuration:', json.dumps(config, indent=2))

import hashlib
initialization = json.loads(Path('experiments/task3/initialization.json').read_text())
for component, expected in initialization['checkpoint_sha256'].items():
    path = Path('pretrained/task2') / component / 'final/best.pt'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, 'Wrong initialization checkpoint: ' + component
    saved = torch.load(path, map_location='cpu', weights_only=True)
    assert not saved['smoke'], 'Task 3 must initialize from trained Task 2 models'
print('Verified four trained Task 2 initialization checkpoints.')


In [ ]:
# Restore previous results after reconnecting, and define consistent backup helpers.
for folder in ['runs', 'models', 'mlruns']:
    if (BACKUP / folder).exists():
        shutil.copytree(BACKUP / folder, PROJECT / folder, dirs_exist_ok=True)
if (BACKUP / 'mlflow.db').exists():
    shutil.copy2(BACKUP / 'mlflow.db', PROJECT / 'mlflow.db')

def backup_results():
    ignore = shutil.ignore_patterns('*.sqlite3', '*.db', '*.tmp', '*-journal', '*-wal', '*-shm')
    for folder in ['runs', 'models', 'mlruns']:
        if (PROJECT / folder).exists():
            shutil.copytree(PROJECT / folder, BACKUP / folder, dirs_exist_ok=True, ignore=ignore)
    databases = list((PROJECT / 'runs').rglob('*.sqlite3')) if (PROJECT / 'runs').exists() else []
    if (PROJECT / 'mlflow.db').exists():
        databases.append(PROJECT / 'mlflow.db')
    for source in databases:
        local_copy = Path('/content') / (source.name + '.backup')
        local_copy.unlink(missing_ok=True)
        with sqlite3.connect(str(source), timeout=60) as src, sqlite3.connect(str(local_copy)) as dest:
            src.backup(dest)
        destination = BACKUP / source.relative_to(PROJECT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_copy, destination)

def run(arguments):
    process = subprocess.Popen([sys.executable, '-u', *arguments], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, cwd=PROJECT)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(record, dict) and 'epoch' in record:
                backup_results()
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        process.terminate()
        process.wait()
        backup_results()
        raise
    backup_results()


In [ ]:
# Download data to local runtime storage; preserve official split definitions.
import urllib.request, tarfile
raw = PROJECT / 'raw_data'
raw.mkdir(exist_ok=True)
for filename in ['102flowers.tgz', 'setid.mat']:
    path = raw / filename
    if not path.exists():
        temporary = raw / (filename + '.part')
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/flowers/102/' + filename, temporary)
        temporary.replace(path)
if not (raw / 'jpg' / 'image_08189.jpg').exists():
    with tarfile.open(raw / '102flowers.tgz') as archive:
        archive.extractall(raw, filter='data')
run(['prepare_dataset.py'])
# Fail before training if preparation differs from the original trained baseline.
import hashlib
baseline_data = json.loads(Path('experiments/task3/baseline_data.json').read_text())
for filename, key in [('train.json', 'train_manifest_sha256'),
                      ('validation_manifest.json', 'validation_manifest_sha256')]:
    manifest = Path('data/prepared_v2/restoration') / filename
    assert hashlib.sha256(manifest.read_bytes()).hexdigest() == baseline_data[key], filename + ' differs from baseline'
print('Training and validation manifests match the baseline.')
run(['-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## Search the soft-gating and joint-loss configuration


In [ ]:
import optuna
SEARCH = Path('runs/task3/search')
FINAL = Path('runs/task3/final')
TRIAL_BUDGET = 6
TRIAL_WARMUP_EPOCHS = 1
TRIAL_JOINT_EPOCHS = 3
FINAL_WARMUP_EPOCHS = 3
FINAL_JOINT_EPOCHS = 20
# Keep settings unchanged when resuming an existing study.
study_path = SEARCH / 'study.sqlite3'
finished = 0
if study_path.exists():
    study = optuna.load_study(study_name='task3', storage='sqlite:///' + str(study_path.resolve()))
    finished = sum(t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED] for t in study.trials)
remaining = max(0, TRIAL_BUDGET - finished)
if remaining:
    run(['-m', 'task3.train', 'tune', '--config', 'configs/task3.json',
         '--initialization', 'pretrained/task2', '--warmup-epochs', str(TRIAL_WARMUP_EPOCHS),
         '--joint-epochs', str(TRIAL_JOINT_EPOCHS), '--trials', str(remaining),
         '--device', 'cuda', '--workers', '2', '--output', str(SEARCH)])
print(json.loads((SEARCH / 'best_config.json').read_text()))


## Warm up the gate, then fine-tune jointly
Both the last completed epoch and best joint-stage checkpoint are backed up. The training code verifies initialization hashes, configuration and data before resuming. A new Adam optimizer is created at the stage transition with the reduced joint learning rate.


In [ ]:
selected = SEARCH / 'best_config.json'
chosen = json.loads(selected.read_text())
chosen['warmup_epochs'] = FINAL_WARMUP_EPOCHS
chosen['joint_epochs'] = FINAL_JOINT_EPOCHS
last = FINAL / 'last.pt'
completed = 0
if last.exists():
    saved = torch.load(last, map_location='cpu', weights_only=True)
    assert saved['config'] == chosen, 'Selected configuration changed; use a separate experiment directory.'
    completed = saved['epoch']
if completed < FINAL_WARMUP_EPOCHS + FINAL_JOINT_EPOCHS:
    arguments = ['-m', 'task3.train', 'train', '--config', str(selected), '--initialization', 'pretrained/task2',
                 '--warmup-epochs', str(FINAL_WARMUP_EPOCHS), '--joint-epochs', str(FINAL_JOINT_EPOCHS),
                 '--device', 'cuda', '--workers', '2', '--output', str(FINAL)]
    if last.exists():
        arguments += ['--resume', str(last)]
    run(arguments)
else:
    assert (FINAL / 'best.pt').exists()
    print('Final training already complete:', completed, 'epochs')


## Compare soft restoration with your fixed Task 2 baseline


In [ ]:
run(['-m', 'task3.evaluate', '--checkpoint', str(FINAL / 'best.pt'), '--initialization', 'pretrained/task2',
     '--split', 'val', '--device', 'cuda', '--output', 'runs/task3/validation'])
run(['-m', 'task3.evaluate', '--checkpoint', str(FINAL / 'best.pt'), '--initialization', 'pretrained/task2',
     '--split', 'val', '--severity-sweep', '--device', 'cuda', '--output', 'runs/task3/severity_validation'])
# This separates gate-only behavior from subsequent expert fine-tuning.
run(['-m', 'task3.evaluate', '--checkpoint', str(FINAL / 'warmup_best.pt'), '--initialization', 'pretrained/task2',
     '--split', 'val', '--device', 'cuda', '--output', 'runs/task3/warmup_validation'])
from IPython.display import display, Image
for folder in ['validation','severity_validation']:
    for name in ['routing_heatmap.png','weight_distributions.png','gate_dominant.png','gate_distributed.png']:
        display(Image(filename=str(Path('runs/task3') / folder / name)))


In [ ]:
import pandas as pd
rows = []
for evaluation, reference_path, result_path in [
    ('original validation', 'experiments/task3/task2_validation.json', 'runs/task3/validation/results.json'),
    ('severity sweep', 'experiments/task3/task2_severity_validation.json', 'runs/task3/severity_validation/results.json')]:
    before = json.loads(Path(reference_path).read_text())
    current = json.loads(Path(result_path).read_text())
    assert current['manifest_sha256'] == before['manifest_sha256'], 'Comparison manifest changed'
    assert current['metrics'].keys() == before['metrics'].keys()
    for group, scores in current['metrics'].items():
        previous = before['metrics'][group]
        assert scores['n'] == previous['n']
        rows.append({'evaluation': evaluation, 'group': group, 'n': scores['n'],
                     'input_psnr': scores['input_psnr'], 'task2_psnr': previous['predicted_psnr'],
                     'soft_psnr': scores['soft_psnr'], 'psnr_change': scores['soft_psnr'] - previous['predicted_psnr'],
                     'task2_ssim': previous['predicted_ssim'], 'soft_ssim': scores['soft_ssim'],
                     'ssim_change': scores['soft_ssim'] - previous['predicted_ssim'],
                     'l1_change': scores['soft_l1'] - previous['predicted_l1']})
comparison = pd.DataFrame(rows)
comparison.to_csv('runs/task3/task2_comparison.csv', index=False)
display(comparison.round(4))
print('Compare clean/mild blur separately. Positive PSNR/SSIM changes and negative L1 changes mean improvement.')
backup_results()


## Export the complete soft mixture as one ONNX graph


In [ ]:
run(['-m', 'task3.export', '--checkpoint', str(FINAL / 'best.pt'), '--output', 'models/task3/soft_moe.onnx'])
from backend.task3_runtime import Task3Runtime
from task3.model import load_checkpoint
from dataset_loaders import RestorationDataset
import numpy as np
runtime = Task3Runtime('models/task3/soft_moe.onnx')
model, _ = load_checkpoint(FINAL / 'best.pt')
model.eval()
with torch.no_grad():
    for condition in ['clean','salt_and_pepper','gaussian_blur','rectangular_occlusion']:
        for item in RestorationDataset(split='val', condition=condition, limit=4):
            x = item['input'][None]
            expected, expected_weights = model(x)
            actual, weights = runtime.run(x.numpy())
            np.testing.assert_allclose(actual, expected.numpy(), rtol=1e-4, atol=1e-5)
            np.testing.assert_allclose(weights, expected_weights.numpy()[0], rtol=1e-4, atol=1e-5)
print('Full ONNX image and weight parity passed on 16 fixed validation cases.')
backup_results()


## Download the model, validation panels and training evidence


In [ ]:
backup_results()
output = Path('/content/task3_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models','runs','mlruns','experiments/task3']:
        for path in (PROJECT / folder).rglob('*'):
            if path.is_file() and not path.name.endswith(('-journal','-wal','-shm','.tmp','.sqlite3')):
                archive.write(path, path.relative_to(PROJECT))
            elif path.is_file() and path.suffix == '.sqlite3':
                archive.write(BACKUP / path.relative_to(PROJECT), path.relative_to(PROJECT))
    archive.write(PROJECT / 'configs/task3.json', 'configs/task3.json')
    if (BACKUP / 'mlflow.db').exists():
        archive.write(BACKUP / 'mlflow.db', 'mlflow.db')
files.download(str(output))
